# 01 — Évaluation MEDCON sur le dictionnaire nettoyé

Ce notebook évalue la qualité de traduction médicale de **GPT-4o** via la métrique **MEDCON groupée** en utilisant le dictionnaire nettoyé (`cleaned_mesh_snomed_dico.json`).

**Pipeline :**
1. Chargement du dictionnaire nettoyé (2 515 entrées)
2. Construction des automates Aho-Corasick EN/FR
3. Calcul MEDCON groupé sur les 49 documents WMT24
4. Calcul BLEU + COMET
5. Comparaison des métriques
6. Analyse des erreurs de terminologie
7. Visualisations

**Fichiers nécessaires dans Google Drive :**
```
TER/
├── data/
│   ├── cleaned_mesh_snomed_dico.json
│   └── corpus_WMT24.json
└── src/
    └── medcon.py
```

## 0. Montage Google Drive & installation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%capture
!pip install pyahocorasick sacrebleu unbabel-comet

## 1. Imports

In [ ]:
import json
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from collections import Counter
from scipy.stats import pearsonr, spearmanr
import ahocorasick
import sacrebleu

# Ajout du dossier src au path pour importer medcon.py
sys.path.insert(0, '/content/drive/MyDrive/TER/src')
from medcon import build_pairs, build_automaton, medcon_grouped

print('Imports OK')

## 2. Chargement du dictionnaire nettoyé

In [ ]:
DICT_PATH   = '/content/drive/MyDrive/TER/data/cleaned_mesh_snomed_dico.json'
CORPUS_PATH = '/content/drive/MyDrive/TER/data/corpus_WMT24.json'

with open(DICT_PATH, 'r', encoding='utf-8') as f:
    raw_dict = json.load(f)

print(f'Dictionnaire nettoyé chargé : {len(raw_dict):,} entrées EN')
print('\n--- Aperçu des 5 premières entrées ---')
for en_term, fr_terms in list(raw_dict.items())[:5]:
    print(f'  {en_term!r:40s} -> {fr_terms}')

## 3. Construction des automates Aho-Corasick

In [ ]:
pairs        = build_pairs(raw_dict)
automaton_en = build_automaton(pairs, 'en')
automaton_fr = build_automaton(pairs, 'fr')

print(f'Paires : {len(pairs):,}  |  Termes EN : {len(automaton_en):,}  |  Termes FR : {len(automaton_fr):,}')

# Test rapide
r_test = medcon_grouped(
    'The patient had a myocardial infarction and was treated for hypertension.',
    'Le patient a subi un infarctus du myocarde et a été traité pour hypertension artérielle.',
    pairs, automaton_en, automaton_fr
)
print(f'\nTest rapide -> P={r_test["precision"]:.2f}  R={r_test["recall"]:.2f}  F1={r_test["f1"]:.2f}')
print(f'Matchées : {r_test["matched"]}')

## 4. Chargement du corpus WMT24

In [ ]:
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    test_docs = json.load(f)

print(f'Corpus : {len(test_docs)} documents')
print(f'Clés   : {list(test_docs[0].keys())}')
print(f'\nExemple doc #0 (source EN) :\n{test_docs[0]["text_en"][:300]}...')

## 5. MEDCON groupé sur GPT-4o

In [ ]:
print('=' * 80)
print('MEDCON GROUPÉ -- GPT-4o  (dictionnaire nettoyé)')
print('=' * 80)

results_gpt = []
for i, doc in enumerate(tqdm(test_docs, desc='MEDCON')):
    r = medcon_grouped(
        doc['text_en'],
        doc['gpt_translation'],
        pairs, automaton_en, automaton_fr
    )
    results_gpt.append({
        'doc_id'           : i,
        'source_en'        : doc['text_en'],
        'gpt_fr'           : doc['gpt_translation'],
        'ref_fr'           : doc.get('translation_fr', ''),
        'medcon_f1'        : r['f1'],
        'medcon_precision' : r['precision'],
        'medcon_recall'    : r['recall'],
        'n_expected'       : r['n_expected'],
        'n_found'          : r['n_found'],
        'n_match'          : r['n_match'],
        'en_concepts'      : r['en_concepts'],
        'matched'          : r['matched'],
        'missed'           : r['missed'],
        'extra'            : r['extra'],
    })

df_gpt = pd.DataFrame(results_gpt)

print(f'\n  Precision : {df_gpt["medcon_precision"].mean():.3f}')
print(f'  Recall    : {df_gpt["medcon_recall"].mean():.3f}')
print(f'  F1        : {df_gpt["medcon_f1"].mean():.3f}')
print(f'\n  Paires attendues (moy.) : {df_gpt["n_expected"].mean():.1f}')
print(f'  Paires trouvées  (moy.) : {df_gpt["n_found"].mean():.1f}')
print(f'  Paires matchées  (moy.) : {df_gpt["n_match"].mean():.1f}')

## 6. BLEU sur GPT-4o

In [ ]:
print('Calcul BLEU...')
for i, doc in enumerate(tqdm(test_docs, desc='BLEU')):
    bleu = sacrebleu.sentence_bleu(doc['gpt_translation'], [doc['translation_fr']]).score
    df_gpt.loc[i, 'bleu'] = bleu

print(f'BLEU moyen : {df_gpt["bleu"].mean():.2f}')

## 7. COMET sur GPT-4o (optionnel — nécessite GPU)

In [ ]:
USE_COMET = False
try:
    import torch
    from comet import download_model, load_from_checkpoint

    print('Chargement COMET (wmt22-comet-da)...')
    comet_model = load_from_checkpoint(download_model('Unbabel/wmt22-comet-da'))

    comet_data = [
        {'src': doc['text_en'], 'mt': doc['gpt_translation'], 'ref': doc['translation_fr']}
        for doc in test_docs
    ]
    comet_output = comet_model.predict(
        comet_data,
        batch_size=8,
        gpus=1 if torch.cuda.is_available() else 0
    )
    for i, score in enumerate(comet_output.scores):
        df_gpt.loc[i, 'comet'] = float(score)

    USE_COMET = True
    print(f'COMET moyen : {df_gpt["comet"].mean():.3f}')

except Exception as e:
    print(f'COMET indisponible : {e}')
    df_gpt['comet'] = np.nan

## 8. Comparaison MEDCON / BLEU / COMET

In [ ]:
print('=' * 80)
print('COMPARAISON MEDCON vs BLEU vs COMET  (GPT-4o)')
print('=' * 80)

print(f"\n{'Métrique':<14} {'Moyenne':>9} {'Std':>9} {'Min':>9} {'Max':>9}")
print('-' * 52)
for name, col in [('MEDCON F1', 'medcon_f1'), ('BLEU', 'bleu'), ('COMET', 'comet')]:
    if col in df_gpt and df_gpt[col].notna().any():
        s = df_gpt[col].dropna()
        print(f'{name:<14} {s.mean():>9.3f} {s.std():>9.3f} {s.min():>9.3f} {s.max():>9.3f}')

print('\n--- Corrélations entre métriques ---')
r_mb, p_mb = pearsonr(df_gpt['medcon_f1'], df_gpt['bleu'])
print(f'  MEDCON <-> BLEU  : r = {r_mb:+.3f}  (p={p_mb:.4f})')
if USE_COMET:
    r_mc, p_mc = pearsonr(df_gpt['medcon_f1'], df_gpt['comet'])
    r_bc, p_bc = pearsonr(df_gpt['bleu'],      df_gpt['comet'])
    print(f'  MEDCON <-> COMET : r = {r_mc:+.3f}  (p={p_mc:.4f})')
    print(f'  BLEU   <-> COMET : r = {r_bc:+.3f}  (p={p_bc:.4f})')

## 9. Analyse des erreurs de terminologie

In [ ]:
print('=' * 80)
print('ANALYSE DES ERREURS DE TERMINOLOGIE -- GPT-4o')
print('=' * 80 + '\n')

all_missed = []
for ml in df_gpt['missed']:
    all_missed.extend(ml)
missed_counter = Counter(all_missed)

print('TOP 30 PAIRES MANQUÉES (terme EN présent en source, terme FR absent en traduction) :')
for i, (lbl, count) in enumerate(missed_counter.most_common(30), 1):
    print(f'  {i:2d}. [{count:2d}x] {lbl}')

all_extra = []
for el in df_gpt['extra']:
    all_extra.extend(el)
extra_counter = Counter(all_extra)

print('\nTOP 30 PAIRES EXTRAS (terme FR dans traduction, terme EN absent de la source) :')
for i, (lbl, count) in enumerate(extra_counter.most_common(30), 1):
    print(f'  {i:2d}. [{count:2d}x] {lbl}')

print(f'\n  Paires manquées uniques : {len(missed_counter)}')
print(f'  Paires extras uniques   : {len(extra_counter)}')

## 10. Exemples détaillés : meilleur / médian / pire document

In [ ]:
def afficher_doc(row, titre):
    print(f"\n{'#'*80}\n  {titre}  (doc #{row['doc_id']})\n{'#'*80}")
    print(f"\nSOURCE EN :\n{row['source_en'][:400]}...")
    print(f"\nTRADUCTION GPT-4o FR :\n{row['gpt_fr'][:400]}...")
    print(f"\n{'─'*60}")
    print(f"MEDCON  F1={row['medcon_f1']:.3f}  P={row['medcon_precision']:.3f}  R={row['medcon_recall']:.3f}")
    if not np.isnan(row.get('bleu', float('nan'))):
        print(f"BLEU    {row['bleu']:.1f}")
    print(f"Paires  attendues={row['n_expected']}  trouvées={row['n_found']}  matchées={row['n_match']}")
    print(f"\n[OK] MATCHÉES ({len(row['matched'])}) :")
    for c in row['matched'][:8]:  print(f'   {c}')
    print(f"\n[X]  MANQUÉES ({len(row['missed'])}) :")
    for c in row['missed'][:8]:   print(f'   {c}')
    print(f"\n[+]  EXTRAS   ({len(row['extra'])}) :")
    for c in row['extra'][:8]:    print(f'   {c}')


best_id   = df_gpt['medcon_f1'].idxmax()
worst_id  = df_gpt['medcon_f1'].idxmin()
median_id = (df_gpt['medcon_f1'] - df_gpt['medcon_f1'].median()).abs().idxmin()

afficher_doc(df_gpt.iloc[best_id],   'MEILLEUR DOCUMENT (F1 max)')
afficher_doc(df_gpt.iloc[median_id], 'DOCUMENT MÉDIAN')
afficher_doc(df_gpt.iloc[worst_id],  'PIRE DOCUMENT (F1 min)')

## 11. Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('MEDCON groupé — GPT-4o (dictionnaire nettoyé)', fontsize=13, fontweight='bold')

# 1. Distribution MEDCON F1
ax = axes[0]
ax.hist(df_gpt['medcon_f1'], bins=20, color='#3498db', edgecolor='black', alpha=0.85)
ax.axvline(df_gpt['medcon_f1'].mean(), color='red', linestyle='--', lw=2,
           label=f"Moy. {df_gpt['medcon_f1'].mean():.3f}")
ax.set_xlabel('MEDCON F1')
ax.set_ylabel('Documents')
ax.set_title('Distribution MEDCON F1')
ax.legend()
ax.grid(alpha=0.3)

# 2. Precision vs Recall
ax = axes[1]
ax.scatter(df_gpt['medcon_recall'], df_gpt['medcon_precision'],
           alpha=0.65, color='#e74c3c', edgecolors='white', s=60)
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision vs Recall (GPT-4o)')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)

# 3. MEDCON vs BLEU scatter
ax = axes[2]
ax.scatter(df_gpt['bleu'], df_gpt['medcon_f1'],
           alpha=0.65, color='#2ecc71', edgecolors='white', s=60)
ax.set_xlabel('BLEU')
ax.set_ylabel('MEDCON F1')
ax.set_title(f'MEDCON vs BLEU  (r={r_mb:.2f})')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/TER/results/01_medcon_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée.')

## 12. Export des résultats

In [ ]:
# Colonnes JSON-sérialisables uniquement
cols_export = ['doc_id', 'medcon_f1', 'medcon_precision', 'medcon_recall',
               'n_expected', 'n_found', 'n_match', 'bleu', 'comet']
cols_export = [c for c in cols_export if c in df_gpt.columns]

out_path = '/content/drive/MyDrive/TER/results/01_medcon_gpt4o.json'
df_gpt[cols_export].to_json(out_path, orient='records', indent=2, force_ascii=False)
print(f'Résultats exportés : {out_path}')

# Résumé final
print('\n' + '=' * 50)
print('RÉSUMÉ FINAL')
print('=' * 50)
print(f"  Dictionnaire : cleaned_mesh_snomed_dico.json ({len(raw_dict):,} entrées)")
print(f"  Corpus       : WMT24 ({len(test_docs)} documents)")
print(f"  MEDCON F1    : {df_gpt['medcon_f1'].mean():.3f} ± {df_gpt['medcon_f1'].std():.3f}")
print(f"  BLEU         : {df_gpt['bleu'].mean():.2f} ± {df_gpt['bleu'].std():.2f}")
if USE_COMET:
    print(f"  COMET        : {df_gpt['comet'].mean():.3f} ± {df_gpt['comet'].std():.3f}")